# Reto 6: Validador de Códigos con Expresiones Regulares

**Nombre:** Diana Alejandra Valdés Luis
**Materia:** Programación para Ciencia de Datos  
**Instituto:** IPN  

---

## Descripción
Este programa valida códigos usando expresiones regulares.

In [1]:
import re
from typing import Dict, List
from datetime import datetime
import csv

In [2]:
DEPARTAMENTOS_VALIDOS = ['VEN', 'ADM', 'TEC', 'LOG', 'RHH']
SERIES_VALIDAS = ['A', 'B', 'C', 'D', 'E']

In [3]:


def validar_producto(codigo: str) -> Dict:
    resultado = {"valido": False, "categoria": None, "numero": None, "pais": None}

    patron = r'^([A-Z]{3})-(\d{4})-([A-Z]{2})$'
    match = re.match(patron, codigo)

    if match:
        resultado["valido"] = True
        resultado["categoria"] = match.group(1)
        resultado["numero"] = match.group(2)
        resultado["pais"] = match.group(3)

    return resultado


def validar_envio(codigo: str) -> Dict:
    resultado = {"valido": False, "fecha": None, "secuencial": None}

    patron = r'^ENV-(20[2-3][0-9])-(0[1-9]|1[0-2])-(0[1-9]|[12][0-9]|3[01])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        anio, mes, dia, sec = match.groups()

        try:
            datetime(int(anio), int(mes), int(dia))
            resultado["valido"] = True
            resultado["fecha"] = f"{anio}-{mes}-{dia}"
            resultado["secuencial"] = sec
        except:
            pass

    return resultado


def validar_empleado(codigo: str) -> Dict:
    resultado = {"valido": False, "departamento": None, "numero": None}

    patron = r'^EMP-([A-Z]{3})-([1-9]\d{3})$'
    match = re.match(patron, codigo)

    if match:
        depto, num = match.groups()

        if depto in DEPARTAMENTOS_VALIDOS:
            resultado["valido"] = True
            resultado["departamento"] = depto
            resultado["numero"] = num

    return resultado


def validar_factura(codigo: str) -> Dict:
    resultado = {"valido": False, "serie": None, "numero": None}

    patron = r'^FAC-([A-E])-(\d{6})$'
    match = re.match(patron, codigo)

    if match:
        serie, num = match.groups()

        if serie in SERIES_VALIDAS:
            resultado["valido"] = True
            resultado["serie"] = serie
            resultado["numero"] = num

    return resultado



def validar_codigo(codigo: str) -> Dict:
    resultado = {
        "codigo": codigo,
        "tipo": "desconocido",
        "valido": False,
        "detalles": {}
    }

    if codigo.startswith("ENV"):
        resultado["tipo"] = "envio"
        res = validar_envio(codigo)
    elif codigo.startswith("EMP"):
        resultado["tipo"] = "empleado"
        res = validar_empleado(codigo)
    elif codigo.startswith("FAC"):
        resultado["tipo"] = "factura"
        res = validar_factura(codigo)
    elif re.match(r'^[A-Z]{3}-', codigo):
        resultado["tipo"] = "producto"
        res = validar_producto(codigo)
    else:
        return resultado

    resultado["valido"] = res["valido"]
    resultado["detalles"] = res

    return resultado


def procesar_lote(codigos: List[str]) -> Dict:
    resultado = {
        "total": 0,
        "validos": 0,
        "invalidos": 0,
        "por_tipo": {
            "producto": {"total": 0, "validos": 0},
            "envio": {"total": 0, "validos": 0},
            "empleado": {"total": 0, "validos": 0},
            "factura": {"total": 0, "validos": 0},
            "desconocido": {"total": 0, "validos": 0}
        },
        "detalle": []
    }

    for codigo in codigos:
        res = validar_codigo(codigo)

        resultado["detalle"].append(res)
        resultado["total"] += 1

        tipo = res["tipo"]
        resultado["por_tipo"][tipo]["total"] += 1

        if res["valido"]:
            resultado["validos"] += 1
            resultado["por_tipo"][tipo]["validos"] += 1
        else:
            resultado["invalidos"] += 1

    return resultado

In [4]:
print("===== PRUEBAS COMPLETAS =====\n")

codigos_prueba = [
    "TEC-0001-MX",
    "ALI-9999-US",
    "ROB-1234-CA",
    "tec-0001-MX",
    "TEC-001-MX",
    "TECH-0001-MX",

    "ENV-2024-03-15-001234",
    "ENV-2025-12-01-999999",
    "ENV-2019-03-15-001234",
    "ENV-2024-13-15-001234",
    "ENV-2024-03-32-001234",

    "EMP-VEN-1234",
    "EMP-TEC-9999",
    "EMP-ADM-1000",
    "EMP-VEN-0123",
    "EMP-XXX-1234",
    "EMP-VEN-123",

    "FAC-A-123456",
    "FAC-E-000001",
    "FAC-B-999999",
    "FAC-F-123456",
    "FAC-A-12345",
    "FAC-a-123456",

    "XXX-1234",
    "RANDOM-CODE",
]

for codigo in codigos_prueba:
    resultado = validar_codigo(codigo)

    estado = "✓" if resultado["valido"] else "✗"
    print(f"{estado} {codigo}  → Tipo: {resultado['tipo']}")

    if resultado["valido"]:
        print("   Detalles:", resultado["detalles"])

    print("-" * 50)

===== PRUEBAS COMPLETAS =====

✓ TEC-0001-MX  → Tipo: producto
   Detalles: {'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}
--------------------------------------------------
✓ ALI-9999-US  → Tipo: producto
   Detalles: {'valido': True, 'categoria': 'ALI', 'numero': '9999', 'pais': 'US'}
--------------------------------------------------
✓ ROB-1234-CA  → Tipo: producto
   Detalles: {'valido': True, 'categoria': 'ROB', 'numero': '1234', 'pais': 'CA'}
--------------------------------------------------
✗ tec-0001-MX  → Tipo: desconocido
--------------------------------------------------
✗ TEC-001-MX  → Tipo: producto
--------------------------------------------------
✗ TECH-0001-MX  → Tipo: desconocido
--------------------------------------------------
✓ ENV-2024-03-15-001234  → Tipo: envio
   Detalles: {'valido': True, 'fecha': '2024-03-15', 'secuencial': '001234'}
--------------------------------------------------
✓ ENV-2025-12-01-999999  → Tipo: envio
   Detalles: 

In [5]:
reporte = procesar_lote(codigos_prueba)

print("===== REPORTE =====\n")
print("Total:", reporte["total"])
print("Válidos:", reporte["validos"])
print("Inválidos:", reporte["invalidos"])

print("\nPor tipo:")
for tipo, stats in reporte["por_tipo"].items():
    print(f"{tipo}: {stats}")

===== REPORTE =====

Total: 25
Válidos: 11
Inválidos: 14

Por tipo:
producto: {'total': 5, 'validos': 3}
envio: {'total': 5, 'validos': 2}
empleado: {'total': 6, 'validos': 3}
factura: {'total': 6, 'validos': 3}
desconocido: {'total': 3, 'validos': 0}


In [7]:
reporte = procesar_lote(codigos_prueba)
print(reporte)

{'total': 25, 'validos': 11, 'invalidos': 14, 'por_tipo': {'producto': {'total': 5, 'validos': 3}, 'envio': {'total': 5, 'validos': 2}, 'empleado': {'total': 6, 'validos': 3}, 'factura': {'total': 6, 'validos': 3}, 'desconocido': {'total': 3, 'validos': 0}}, 'detalle': [{'codigo': 'TEC-0001-MX', 'tipo': 'producto', 'valido': True, 'detalles': {'valido': True, 'categoria': 'TEC', 'numero': '0001', 'pais': 'MX'}}, {'codigo': 'ALI-9999-US', 'tipo': 'producto', 'valido': True, 'detalles': {'valido': True, 'categoria': 'ALI', 'numero': '9999', 'pais': 'US'}}, {'codigo': 'ROB-1234-CA', 'tipo': 'producto', 'valido': True, 'detalles': {'valido': True, 'categoria': 'ROB', 'numero': '1234', 'pais': 'CA'}}, {'codigo': 'tec-0001-MX', 'tipo': 'desconocido', 'valido': False, 'detalles': {}}, {'codigo': 'TEC-001-MX', 'tipo': 'producto', 'valido': False, 'detalles': {'valido': False, 'categoria': None, 'numero': None, 'pais': None}}, {'codigo': 'TECH-0001-MX', 'tipo': 'desconocido', 'valido': False, '

## Conclusión

Aprendí a usar expresiones regulares para validar datos y a estructurar funciones en Python.

Esto puede aplicarse en sistemas reales como validación de formularios o bases de datos.

La actividad ayudó a mejorar mi lógica de programación.